<a href="https://colab.research.google.com/github/coreprimejio/ev-server/blob/master-qa/data_handling_(Corrected).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import json
import os
import uuid
import shutil
import matplotlib.pyplot as plt
from pylatex import Document, Command, Enumerate
from pylatex.utils import NoEscape

# --- Graphing Functions using Matplotlib ---

def create_bar_graph(categories, values, xlabel, ylabel, output_filename):
    """Generates a bar graph from categories and values."""
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(categories, values, color='cornflowerblue', edgecolor='black', width=0.6)
    ax.set_xlabel(xlabel, fontsize=12, weight='bold')
    ax.set_ylabel(ylabel, fontsize=12, weight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    # Add value labels on top of each bar
    for i, v in enumerate(values):
        ax.text(i, v + 0.5, str(v), ha='center', va='bottom')
    plt.tight_layout()
    plt.savefig(output_filename, format='png', dpi=150, bbox_inches='tight', pad_inches=0.1)
    plt.close()

def create_line_graph(x_points, y_points, xlabel, ylabel, output_filename):
    """Generates a line graph from x and y points."""
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(x_points, y_points, marker='o', color='mediumseagreen', linestyle='-')
    ax.set_xlabel(xlabel, fontsize=12, weight='bold')
    ax.set_ylabel(ylabel, fontsize=12, weight='bold')
    ax.grid(True, linestyle='--', alpha=0.7)
    # Add value labels at each point
    for (x, y) in zip(x_points, y_points):
        ax.text(x, y + (max(y_points)*0.02), str(y), ha='center')
    plt.tight_layout()
    plt.savefig(output_filename, format='png', dpi=150, bbox_inches='tight', pad_inches=0.1)
    plt.close()

# --- PDF GENERATION SCRIPT ---
def generate_quiz_pdf(quiz_data, pdf_filepath):
    """Reads a JSON object of questions and generates a single-column A4 PDF."""
    geometry_options = {"tmargin": "1in", "lmargin": "1in"}
    doc = Document(pdf_filepath, documentclass='article', document_options=['a4paper', '12pt'], geometry_options=geometry_options)

    doc.preamble.append(Command('usepackage', 'graphicx'))
    doc.preamble.append(Command('usepackage', 'enumitem'))
    doc.preamble.append(Command('usepackage', 'xcolor'))
    doc.preamble.append(Command('setlength', [NoEscape(r'\parindent'), '0pt']))
    doc.preamble.append(NoEscape(r'\definecolor{questioncolor}{RGB}{40,80,150}'))

    title = quiz_data.get('topic', 'Quiz')
    class_level = quiz_data.get('class', '')
    module_name = quiz_data.get('module', '')

    doc.append(NoEscape(r'\begin{center}'))
    doc.append(NoEscape(r'{\Huge\bfseries ' + title + r'}\\[5pt]'))
    doc.append(NoEscape(r'{\Large\bfseries ' + module_name + r'}\\[10pt]'))
    doc.append(NoEscape(r'{\large ' + class_level + r'}'))
    doc.append(NoEscape(r'\end{center}\bigskip'))

    output_pdf_dir = os.path.dirname(pdf_filepath)
    figures_dir = os.path.join(output_pdf_dir, 'figures')
    os.makedirs(figures_dir, exist_ok=True)

    for i, q in enumerate(quiz_data['questions']):
        figure_name = q.get('figure_id', f'q_{i+1}')
        fig_name_only = f'{figure_name}.png' # Using PNG for better quality
        fig_full_path = os.path.join(figures_dir, fig_name_only)

        doc.append(NoEscape(r'\textcolor{questioncolor}{\textbf{Question ' + str(i+1) + r':}} ')); doc.append(NoEscape(q['question']))
        doc.append(NoEscape(r'\par'))

        if 'figure_desc' in q:
            fig_desc = q['figure_desc']

            if not os.path.exists(fig_full_path):
                fig_type = fig_desc.get('type_of_figure')

                if fig_type == 'bar_graph':
                    create_bar_graph(fig_desc['categories'], fig_desc['values'], fig_desc['xlabel'], fig_desc['ylabel'], fig_full_path)
                elif fig_type == 'line_graph':
                    create_line_graph(fig_desc['x_points'], fig_desc['y_points'], fig_desc['xlabel'], fig_desc['ylabel'], fig_full_path)

            if os.path.exists(fig_full_path):
                fig_relative_path = os.path.join('figures', fig_name_only)
                doc.append(NoEscape(r'\begin{center}'))
                doc.append(NoEscape(r'\includegraphics[width=0.75\linewidth, keepaspectratio]{' + fig_relative_path.replace('\\', '/') + '}'))
                doc.append(NoEscape(r'\end{center}'))

        with doc.create(Enumerate(options=NoEscape(r'label=(\alph*)'))) as enum:
            for option in q['options']:
                enum.add_item(NoEscape(option))

        doc.append(NoEscape(r'\bigskip\hrule\bigskip'))

    try:
        doc.generate_pdf(clean_tex=True)
        print(f"✅ Successfully generated PDF: {pdf_filepath}.pdf")
    except Exception as e:
        print(f"❌ PDF generation failed. Ensure a LaTeX distribution is installed. Error: {e}")

if __name__ == '__main__':
    input_dir = '/Users/ajaygupta/class5/fractions/json'
    output_dir = '/Users/ajaygupta/class5/fractions/pdfs'
    image_assets_dir = '/Users/ajaygupta/Downloads/images' # <-- NEW: Name of the folder for your symbol images
    json_file_name = 'fractions.json'

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    json_file_path = os.path.join(input_dir, json_file_name)

    if not os.path.exists(json_file_path):
        print(f"File not found: {json_file_path}. Please create it with your quiz data.")
    else:
        pdf_filename = f"{os.path.splitext(json_file_name)[0]}-{uuid.uuid4()}"
        pdf_full_path = os.path.join(output_dir, pdf_filename)

        try:
            with open(json_file_path, 'r', encoding='utf-8') as f:
                quiz_json_object = json.load(f)
            generate_quiz_pdf(quiz_json_object, pdf_full_path)
        except FileNotFoundError:
            print(f"❌ Error: The file was not found at {json_file_path}")
        except json.JSONDecodeError:
            print(f"❌ Error: The file at {json_file_path} is not a valid JSON file.")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

✅ Successfully generated PDF: /Users/ajaygupta/class5/fractions/pdfs/fractions-6f028952-bb2f-4541-a1d9-53ef6d3223f1.pdf
